# LPatchTST — Kaggle 2x T4 GPU Notebook

This notebook handles repository setup, data preparation, multi-GPU DDP training, and downstream evaluation on Kaggle.

### Pre-requisites:
1. **Internet** must be turned **ON** in the right-hand settings panel.
2. **Accelerator** must be set to **GPU T4 x2**.

In [2]:
%%bash
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Clone Repo & Set Up CSV Dataset                           ║
# ╚══════════════════════════════════════════════════════════════════════╝
REPO_DIR="/kaggle/working/Lpatchtst"

if [ -d "$REPO_DIR" ]; then
    echo "Repository already exists. Updating to latest main..."
    cd "$REPO_DIR"
    git fetch origin
    git reset --hard origin/main
    git submodule update --init --recursive
else
    echo "Cloning repository (including submodules)..."
    git clone --recurse-submodules https://github.com/ayan1-git/Lpatchtst "$REPO_DIR"
fi

KRONOS_DIR="/kaggle/working/Kronos_finetune"
if [ -d "$KRONOS_DIR" ]; then
    echo "Kronos_finetune repository already exists. Updating..."
    cd "$KRONOS_DIR"
    git fetch origin
    git reset --hard origin/main
else
    echo "Cloning Kronos_finetune repository..."
    git clone https://github.com/ayan1-git/Kronos_finetune "$KRONOS_DIR"
fi

# Install lightweight required libraries (torch/numpy/pandas are pre-installed)
pip install -q einops safetensors scikit-learn

cd "$REPO_DIR"
mkdir -p Data

# Check if the cloned repository already contains CSV files in Data/
csv_count=$(ls Data/*.csv 2>/dev/null | wc -l)

if [ "$csv_count" -gt 0 ]; then
    echo "Found $csv_count pre-packaged CSV files in Data/ directory (cloned from Git)."
    echo "Using pre-packaged dataset for training."
    ls -lh Data/
else
    # Auto-discover any Kaggle input folder containing CSV files and link them
    echo "No CSV files found in cloned Data/ folder. Searching under /kaggle/input/..."
    FOUND=0
    for d in /kaggle/input/*; do
        if [ -d "$d" ]; then
            if ls "$d"/*.csv >/dev/null 2>&1; then
                echo "-> Found CSV files in $d. Symlinking to Data/..."
                ln -sf "$d"/*.csv Data/
                FOUND=1
            fi
        fi
    done
    if [ $FOUND -eq 0 ]; then
        echo "⚠️ Warning: No CSV files found. Please copy your CSVs to $REPO_DIR/Data/."
    else
        echo "Data folder contents successfully mapped from Kaggle inputs:"
        ls -lh Data/
    fi
fi
echo "✅ Setup complete!"

Cloning repository (including submodules)...
Cloning Kronos_finetune repository...
Found 19 pre-packaged CSV files in Data/ directory (cloned from Git).
Using pre-packaged dataset for training.
total 28M
-rw-r--r-- 1 root root 1.9M Jun  2 10:49 NIFTY 100_30minute.csv
-rw-r--r-- 1 root root 1.9M Jun  2 10:49 NIFTY 200_30minute.csv
-rw-r--r-- 1 root root 1.2M Jun  2 10:49 NIFTY 500_30minute.csv
-rw-r--r-- 1 root root 1.9M Jun  2 10:49 NIFTY 50_30minute.csv
-rw-r--r-- 1 root root 1.2M Jun  2 10:49 NIFTY ALPHA 50_30minute.csv
-rw-r--r-- 1 root root 1.9M Jun  2 10:49 NIFTY AUTO_30minute.csv
-rw-r--r-- 1 root root 2.0M Jun  2 10:49 NIFTY BANK_30minute (1).csv
-rw-r--r-- 1 root root 1.7M Jun  2 10:49 NIFTY COMMODITIES_30minute.csv
-rw-r--r-- 1 root root 633K Jun  2 10:49 NIFTY CONSR DURBL_30minute.csv
-rw-r--r-- 1 root root 1.4M Jun  2 10:49 NIFTY CONSUMPTION_30minute.csv
-rw-r--r-- 1 root root 1.1M Jun  2 10:49 NIFTY CPSE_30minute.csv
-rw-r--r-- 1 root root 2.0M Jun  2 10:49 NIFTY ENERGY_30m

Cloning into '/kaggle/working/Lpatchtst'...
Cloning into '/kaggle/working/Kronos_finetune'...


In [4]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Run DDP Training on 2x T4 GPUs via torchrun               ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, subprocess, sys, torch

REPO_DIR = "/kaggle/working/Lpatchtst"
os.chdir(REPO_DIR)

# 1. Force unbuffered output and resolve NCCL binding hangs in subprocesses
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["NCCL_SOCKET_IFNAME"] = "lo"

num_gpus = torch.cuda.device_count()
print(f"Detecting GPUs: {num_gpus} active")

cmd = f"torchrun --standalone --nnodes=1 --nproc_per_node={num_gpus} --master_port=29500 finetune_tokenizer.py"
print(f"Running command: {cmd}\n")
print("-" * 70)

# 2. Launch process and stream stdout+stderr line-by-line in real-time
process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
    cwd=REPO_DIR
)

try:
    while True:
        line = process.stdout.readline()
        if not line and process.poll() is not None:
            break
        if line:
            print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n⚠️ Interrupted! Terminating training process…")
    process.terminate()
    process.wait()

rc = process.poll()
if rc != 0:
    raise RuntimeError(f"Training failed with exit code {rc}")
print("\n✅ Training complete successfully!")

Detecting GPUs: 2 active
Running command: torchrun --standalone --nnodes=1 --nproc_per_node=2 --master_port=29500 finetune_tokenizer.py

----------------------------------------------------------------------
W0602 10:49:58.277000 180 torch/distributed/run.py:852] 
W0602 10:49:58.277000 180 torch/distributed/run.py:852] *****************************************
W0602 10:49:58.277000 180 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0602 10:49:58.277000 180 torch/distributed/run.py:852] *****************************************
[W602 10:49:58.359221820 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W602 10:50:01.369169236 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W602 10:50:01.369486863 socket.cpp:207] [c10d] The h

In [5]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Downstream Evaluation & Diagnostics                       ║
# ╚══════════════════════════════════════════════════════════════════════╝
import os, subprocess, sys

REPO_DIR = "/kaggle/working/Lpatchtst"
os.chdir(REPO_DIR)

cmd = "python3 -u audit_training.py"
print(f"Running command: {cmd}\n")
print("-" * 70)

process = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR
)

try:
    while True:
        line = process.stdout.readline()
        if not line and process.poll() is not None:
            break
        if line:
            print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n⚠️ Interrupted! Terminating evaluation process…")
    process.terminate()
    process.wait()

rc = process.poll()
if rc != 0:
    raise RuntimeError(f"Evaluation failed with exit code {rc}")
print("\n✅ Evaluation complete successfully!")

Running command: python3 -u audit_training.py

----------------------------------------------------------------------

LPatchTST Audit  (tokens_only mode)
Repo : /kaggle/working/Lpatchtst
Out  : /kaggle/working/Lpatchtst/audit_output

SECTION 1 — ORACLE TARGET DISTRIBUTION
2026-06-02 11:15:12,576 | INFO     | features | Building features for 'close' | 15717 rows | ohlc=True.
2026-06-02 11:15:12,894 | WARNING  | features | session_cyclic_features: DatetimeIndex is tz-naive. Assuming timestamps are already in Asia/Kolkata. If your data is UTC or another timezone, localize before calling: index = index.tz_localize('UTC').tz_convert('Asia/Kolkata')
2026-06-02 11:15:12,987 | INFO     | features | Feature matrix built: shape=(15717, 20) | NaN count=11641
Target Distribution — Long: 0.187 | Short: 0.131 | Zero: 0.682
[WARN] [ORACLE] Target balance  →  Long=18.82%  Short=13.25%  Zero/Flat=67.93%
[PASS] [ORACLE] Sub-threshold leakage after zeroing  →  0 nonzero targets with |tgt| < SAMPLER_THRE

In [9]:
"""
push_to_hub.py
──────────────
Run this from Kaggle (or locally) after training to upload model
checkpoints + config to HuggingFace Hub.

Usage (Kaggle notebook cell):
    !python /kaggle/working/Lpatchtst/push_to_hub.py

Or set HF_TOKEN as a Kaggle Secret instead of hard-coding it here.
"""

import os
import sys
import json
import shutil
import importlib
import torch

# ── Config ───────────────────────────────────────────────────────────────────
HF_USERNAME   = "gulnawaz123"
REPO_NAME     = "Full_Tokenizer_30m_1"          # change freely
REPO_ID       = f"{HF_USERNAME}/{REPO_NAME}"

# Token: read from environment variable (Kaggle Secret or shell export).
# Set via:  os.environ["HF_TOKEN"] = "hf_..."  in a prior cell, or
#           Add a Kaggle Secret named HF_TOKEN.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    raise EnvironmentError("HF_TOKEN environment variable is not set. "
                           "Add it as a Kaggle Secret or export it in your shell.")

# Paths to look for checkpoints (adjust if your working dir differs)
SEARCH_DIRS = [
    "/kaggle/working/Lpatchtst",
    "/kaggle/working",
    ".",
]
CHECKPOINT_FILES = [
    "pretrain_best.pth",
    "model.safetensors",
]
CONFIG_FILES = [
    "config.py",
    "model.py",
    "loss.py",
    "tokenizer.py",
    "features.py",
    "oracle.py",
    "data_loader.py",
]

# ── Install huggingface_hub if missing ───────────────────────────────────────
try:
    from huggingface_hub import HfApi, create_repo, upload_folder
except ImportError:
    print("Installing huggingface_hub …")
    os.system(f"{sys.executable} -m pip install -q huggingface_hub")
    from huggingface_hub import HfApi, create_repo, upload_folder

# ── Resolve source directory ─────────────────────────────────────────────────
def _find_file(fname):
    for d in SEARCH_DIRS:
        if not os.path.exists(d):
            continue
        for root, dirs, files in os.walk(d):
            # Prune directories we don't want to traverse to keep it fast
            dirs[:] = [name for name in dirs if name not in (
                'data', 'Data', '__pycache__', '.git', '.cursor', '.kilo', '.ipynb_checkpoints'
            ) and not name.startswith('.')]
            if fname in files:
                return os.path.join(root, fname)
    return None

src_dir = None
for d in SEARCH_DIRS:
    if os.path.exists(os.path.join(d, "config.py")):
        src_dir = d
        break

if src_dir is None:
    # Try recursive find if not found directly
    config_path = _find_file("config.py")
    if config_path:
        src_dir = os.path.dirname(config_path)

if src_dir is None:
    raise FileNotFoundError("Cannot locate config.py in any search directory.")

print(f"Source directory : {src_dir}")

# ── Stage files into a temp upload folder ────────────────────────────────────
upload_dir = "/tmp/lpatchtst_hf_upload"
if os.path.exists(upload_dir):
    shutil.rmtree(upload_dir)
os.makedirs(upload_dir, exist_ok=True)

copied = []

# Copy checkpoints
for ckpt in CHECKPOINT_FILES:
    path = _find_file(ckpt)
    if path:
        shutil.copy2(path, os.path.join(upload_dir, ckpt))
        copied.append(ckpt)
        print(f"  ✓ {ckpt}  ({os.path.getsize(path)/1e6:.1f} MB)")
        
        # If this is model.safetensors, also copy any associated config or vocab files
        # in the same directory (necessary for HuggingFace model load)
        if ckpt == "model.safetensors":
            ckpt_dir = os.path.dirname(path)
            for f in os.listdir(ckpt_dir):
                if f != "model.safetensors" and os.path.isfile(os.path.join(ckpt_dir, f)):
                    shutil.copy2(os.path.join(ckpt_dir, f), os.path.join(upload_dir, f))
                    print(f"  ✓ {f} (copied config/vocab file from checkpoint dir)")
    else:
        print(f"  ✗ {ckpt} not found — skipping")

if not copied:
    raise FileNotFoundError(
        "No checkpoint files found. Train the model first.\n"
        f"Expected one of: {CHECKPOINT_FILES}"
    )

# Copy source files
for fname in CONFIG_FILES:
    path = _find_file(fname)
    if path:
        shutil.copy2(path, os.path.join(upload_dir, fname))
        print(f"  ✓ {fname}")

# ── Write a minimal model card ───────────────────────────────────────────────
model_card = f"""---
language:
  - en
tags:
  - time-series
  - finance
  - pytorch
  - patchtst
license: mit
---

# LPatchTST — NIFTY 50 Trading Model

A patch-based Transformer (LPatchTST) trained on NIFTY 50 30-minute bars
for directional signal prediction.

## Files
| File | Description |
|------|-------------|
| `best_model_lpatchtst.pth` | Best fine-tuned checkpoint |
| `pretrained_lpatchtst.pth` | Pre-trained backbone checkpoint |
| `config.py` | Hyperparameters & architecture config |
| `model.py` | Model definition |

## Loading the model
```python
import torch
import sys
sys.path.insert(0, ".")   # ensure local modules are importable

import config
from model import LPatchTST

net = LPatchTST(
    input_mode=config.INPUT_MODE,
    seq_len=config.LOOKBACK_WINDOW,
    n_features=0,          # set to your feature count
    s1_bits=config.TOKENIZER_S1_BITS,
    s2_bits=config.TOKENIZER_S2_BITS,
    d_model=config.D_MODEL,
    patch_len=config.PATCH_LEN,
    stride=config.STRIDE,
    n_heads=config.N_HEADS,
    n_layers=config.N_LAYERS,
    lstm_layers=config.LSTM_LAYERS,
    dropout=config.FINETUNE_DROPOUT,
    aggregation=config.AGGREGATION_MODE,
)
state = torch.load("best_model_lpatchtst.pth", map_location="cpu")
net.load_state_dict(state)
net.eval()
```
"""
with open(os.path.join(upload_dir, "README.md"), "w") as f:
    f.write(model_card)

# ── Create / ensure repo exists ───────────────────────────────────────────────
api = HfApi(token=HF_TOKEN)

print(f"\nCreating / verifying repo: {REPO_ID} …")
create_repo(
    repo_id=REPO_ID,
    token=HF_TOKEN,
    repo_type="model",
    exist_ok=True,
    private=False,      # set True if you want a private repo
)
print(f"  ✓ Repo ready: https://huggingface.co/{REPO_ID}")

# ── Upload ────────────────────────────────────────────────────────────────────
print(f"\nUploading {len(os.listdir(upload_dir))} files …")
api.upload_folder(
    folder_path=upload_dir,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Upload LPatchTST checkpoint and source",
)

print(f"\n✅ Upload complete!")
print(f"   View at: https://huggingface.co/{REPO_ID}")


Source directory : /kaggle/working/Lpatchtst
  ✗ pretrain_best.pth not found — skipping
  ✓ model.safetensors  (15.8 MB)
  ✓ README.md (copied config/vocab file from checkpoint dir)
  ✓ config.json (copied config/vocab file from checkpoint dir)
  ✓ config.py
  ✓ model.py
  ✓ loss.py
  ✓ tokenizer.py
  ✓ features.py
  ✓ oracle.py
  ✓ data_loader.py

Creating / verifying repo: gulnawaz123/Full_Tokenizer_30m_1 …
  ✓ Repo ready: https://huggingface.co/gulnawaz123/Full_Tokenizer_30m_1

Uploading 10 files …


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Upload complete!
   View at: https://huggingface.co/gulnawaz123/Full_Tokenizer_30m_1
